# Day 1: 环境搭建与项目概览

## 🎯 本节目标

完成今天的学习后，你将能够：

1. ✅ 配置可运行的 Python 开发环境（本地或 Colab）
2. ✅ 理解 Marionette 项目的整体架构
3. ✅ 掌握核心模块的作用和关系
4. ✅ 理解配置系统的工作原理

---

## 📚 前端开发者指南

如果你熟悉前端开发，可以用以下类比来理解今天的内容：

| 前端概念 | Marionette 对应概念 |
|---------|---------------------|
| `package.json` | `config/` 目录 + `requirements.txt` |
| `npm install` | `pip install` |
| `webpack.config.js` | `config/train.yaml` |
| `src/` 目录 | `add_thin/` + `discrete_diffusion/` |
| `index.js` 入口 | `train.py` 入口 |
| React 组件树 | 模型实例化流程 |

---

## Part 1: 环境配置

### 1.1 选择你的开发环境

你有两个选择：

**选项 A: Google Colab（推荐初学者）**
- 优点：无需配置，免费 GPU，即开即用
- 缺点：需要上传数据，会话时间限制

**选项 B: 本地环境（推荐进阶）**
- 优点：完全控制，数据持久化
- 缺点：需要自己配置环境

---

### 1.2 Google Colab 配置（推荐）

#### 步骤 1: 打开 Colab

1. 访问 [colab.research.google.com](https://colab.research.google.com/)
2. 点击 "新建笔记本"

#### 步骤 2: 启用 GPU

1. 点击菜单栏 **"Runtime"（运行时）**
2. 选择 **"Change runtime type"（更改运行时类型）**
3. 在 **"Hardware accelerator"（硬件加速器）** 中选择 **"GPU"**
4. 点击 **"Save"（保存）**

In [ ]:
# 检查 GPU 是否可用
import torch

if torch.cuda.is_available():
    print("✅ GPU 可用！\n")
    print(f"GPU 名称: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"GPU 显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  GPU 不可用，将使用 CPU（训练会很慢）")
    print("提示：请按照上面的步骤启用 GPU！")

✅ GPU 可用！

GPU 名称: Tesla T4
CUDA 版本: 12.8
GPU 显存: 14.6 GB


#### 步骤 3: 安装依赖

运行以下单元格安装所有必需的包（大约需要 3-5 分钟）：

In [ ]:
# 1. 彻底卸载冲突源
!pip uninstall -y numpy scipy pandas torch torchvision pytorch-lightning opencv-python jax jaxlib cupy-cuda12x --quiet

# 2. 按照“从底层到高层”的顺序安装
# 先装基础库
!pip install numpy==1.26.4 scipy==1.11.4 --quiet
# 再装框架
!pip install torch==2.2.2 torchvision==0.17.2 --extra-index-url https://download.pytorch.org/whl/cu121 --quiet
# 最后装 Lightning 和其他
!pip install pytorch-lightning==1.9.5 pandas==2.2.2 einops rich wandb hydra-core torchtyping typeguard==4.0.1 pyyaml --quiet

# 3. 验证并强制重启
import numpy as np
import torch
print(f"NumPy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")

import os
os.kill(os.getpid(), 9) # 这一步极其重要，强制系统重新加载 DLL/SO 文件

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchmetrics 1.9.0 requires torch>=2.0.0, which is not installed.
cucim-cu12 26.2.0 requires cupy-cuda12x>=13.6.0, which is not installed.
geemap 0.35.3 requires pandas, which is not installed.
holoviews 1.22.1 requires pandas>=1.3, which is not installed.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, which is not installed.
sentence-transformers 5.2.3 requires torch>=1.11.0, which is not installed.
mlxtend 0.23.4 requires pandas>=0.24.2, which is not installed.
orbax-checkpoint 0.11.33 requires jax>=0.6.0, which is not installed.
fastai 2.8.7 requires pandas, which is not installed.
fastai 2.8.7 requires torch<3,>=1.10, which is not installed.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
pointpats 2.5.5 requires pandas>=2.2, which is not installed.
dopamine-rl 4.1.2 requires jax>=0.1.72, which

In [ ]:
import numpy as np
import torch
import pytorch_lightning as pl

print(f"✅ NumPy 版本: {np.__version__}")
print(f"✅ PyTorch 版本: {torch.__version__}")
print(f"✅ Lightning 版本: {pl.__version__}")

# 测试是否还会报错
try:
    arr = np.array([1, 2, 3])
    print("🚀 NumPy 数组测试成功，无二进制冲突！")
except Exception as e:
    print(f"❌ 仍然报错: {e}")

✅ NumPy 版本: 1.26.4
✅ PyTorch 版本: 2.2.2+cu121
✅ Lightning 版本: 1.9.5
🚀 NumPy 数组测试成功，无二进制冲突！


#### 步骤 4: 挂载项目代码

**方法 1: 从 GitHub 克隆（推荐）**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/um-marionette

import os
os.chdir('/content/drive/MyDrive/um-marionette')

!git clone https://github.com/fengnzl/um-marionette.git

!ls /content/drive/MyDrive/um-marionette/

In [ ]:
# 检查是否在 Colab 中
import os

if os.path.exists('/content'):
    print("检测到 Colab 环境")
    print("\n请选择一种方式加载项目代码：\n")
    print("方法 1: 从 GitHub 克隆")
    print("  !git clone <项目仓库地址>")
    print("\n方法 2: 上传到 Google Drive")
    print("  1. 将项目文件上传到 Google Drive")
    print("  2. 运行: from google.colab import drive")
    print("  3. 运行: drive.mount('/content/drive')")
else:
    print("✅ 本地环境，项目文件已存在")
    print("当前工作目录:", os.getcwd())

检测到 Colab 环境

请选择一种方式加载项目代码：

方法 1: 从 GitHub 克隆
  !git clone <项目仓库地址>

方法 2: 上传到 Google Drive
  1. 将项目文件上传到 Google Drive
  2. 运行: from google.colab import drive
  3. 运行: drive.mount('/content/drive')


In [ ]:
# 1. 定义目标路径
import os
target_path = '/content/drive/MyDrive/um-marionette'

# 2. 如果文件夹不存在则创建（防止路径报错）
if not os.path.exists(target_path):
    os.makedirs(target_path)
    print(f"创建了新目录: {target_path}")

# 3. 切换当前工作目录 (相当于执行了 cd)
os.chdir(target_path)

# 4. 验证当前路径
print(f"✅ 当前工作目录已固定为: {os.getcwd()}")

# 5. 查看该目录下的文件
!ls

Cloning into 'um-marionette'...
remote: Enumerating objects: 81, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 81 (delta 6), reused 72 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (81/81), 114.58 KiB | 1.85 MiB/s, done.
Resolving deltas: 100% (6/6), done.


---

### 1.3 本地环境配置

如果你选择在本地（macOS）开发，请按以下步骤操作：

#### 步骤 1: 安装 Homebrew（如果还没有）

```bash
/bin/bash -c "$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)"
```

#### 步骤 2: 安装 Python 3.10

```bash
brew install python@3.10
```

#### 步骤 3: 创建虚拟环境（类似 npm workspace）

```bash
# 进入项目目录
cd /path/to/Marionette

# 创建虚拟环境
python3.10 -m venv marionette_env

# 激活环境
source marionette_env/bin/activate
```

#### 步骤 4: 安装依赖

```bash
# 安装 PyTorch（如果有 GPU，选择 CUDA 版本）
pip install torch==2.0.0

# 安装其他依赖
pip install -r requirements.txt
```

#### 步骤 5: 安装 Jupyter

```bash
pip install jupyter
jupyter notebook
```

---

## Part 2: Marionette 项目架构

### 2.1 什么是 Marionette？

**Marionette** 是一个基于 **KDD'25** 顶会论文的科研项目。

#### 核心功能：生成人类轨迹数据

想象一个场景：
- 你需要测试一个位置推荐算法
- 但真实的用户数据涉及隐私问题
- **Marionette** 可以生成"假"但"真实"的用户轨迹数据

#### 前端类比

| 前端场景 | Marionette 场景 |
|---------|---------------|
| Mock 数据生成器 | 轨迹数据生成器 |
| 根据 API 返回 Mock 数据 | 根据条件生成轨迹 |
| JSON 格式的 Mock 数据 | 序列化的轨迹数据 |

---

### 2.2 项目目录结构

让我们探索项目的目录结构：

In [ ]:
import os

def show_tree(path, level=0, max_level=2):
    """显示目录结构（类似 tree 命令）"""
    if level > max_level:
        return

    try:
        items = sorted(os.listdir(path))
        for item in items:
            if item.startswith('.'):
                continue
            item_path = os.path.join(path, item)
            prefix = "│   " * level + "├── "

            if os.path.isdir(item_path):
                print(prefix + item + "/")
                show_tree(item_path, level + 1, max_level)
            else:
                print(prefix + item)
    except PermissionError:
        pass

print("📁 Marionette 项目目录结构：")
print("="*50)
show_tree(".", level=0, max_level=2)

### 2.3 核心模块详解

#### 前端类比理解

想象一个 React 应用的结构：

```
src/
├── components/     # UI 组件
├── utils/          # 工具函数
├── hooks/          # 自定义 Hooks
├── App.js          # 主应用
└── index.js        # 入口文件
```

Marionette 的结构类似：

```
Marionette/
├── add_thin/              # 时间模型（组件）
├── discrete_diffusion/    # 空间模型（组件）
├── datamodule.py          # 数据管理（状态管理）
├── configs.py             # 配置工厂（Context）
├── tasks.py               # 训练任务（业务逻辑）
├── train.py               # 训练入口（index.js）
└── config/                # 配置文件（.env）
```

---

#### 模块功能对照表

| 模块 | 功能 | 前端类比 | 关键文件 |
|------|------|---------|----------|
| **add_thin/** | 处理时间维度 | 时间轴组件 | `diffusion/model.py` |
| **discrete_diffusion/** | 处理空间位置 | 地图组件 | `diffusion_transformer.py` |
| **datamodule.py** | 数据加载和批次处理 | Redux/Context | `Sequence`, `Batch` 类 |
| **train.py** | 训练入口 | `index.js` | 主训练循环 |
| **sample.py** | 生成新轨迹 | API 调用 | 采样流程 |
| **configs.py** | 配置管理 | `.env` + Context | 工厂函数 |

---

### 2.4 混合架构设计

Marionette 采用**混合扩散模型**，这是它的核心创新。

#### 为什么要分开建模？

想象一个用户的移动轨迹：

```json
{
  "user_id": "12345",
  "movements": [
    { "time": "2024-01-15 08:00", "place": "Home", "location": [41.0, 28.9] },
    { "time": "2024-01-15 09:30", "place": "Cafe", "location": [41.1, 29.0] },
    { "time": "2024-01-15 12:00", "place": "Office", "location": [41.2, 29.1] }
  ]
}
```

这个轨迹包含**两类不同性质的数据**：

1. **时间**：连续值，有顺序依赖（08:00 → 09:30 → 12:00）
2. **地点**：离散值，类别依赖（Home → Cafe → Office）

#### 前端类比

| 数据类型 | 示例 | 前端处理 | AI 处理 |
|---------|------|---------|--------|
| 时间（连续） | 3600 秒 | `Date.now()` | Add-THIN |
| 地点（离散） | POI ID: 123 | `useState(id)` | 离散扩散 |

```python
# 伪代码：轨迹生成 = 时间生成 + 地点生成
def generate_trajectory(conditions):
    # 1. 生成时间序列（连续值）
    times = add_thin_model.sample(conditions)
    
    # 2. 生成地点序列（离散值）
    places = discrete_diffusion_model.sample(times, conditions)
    
    return combine(times, places)
```

---

## Part 3: 配置系统深度解析

### 3.1 为什么需要配置系统？

#### 前端类比

在 React 项目中，你可能有：

```javascript
// config.js
export const config = {
  API_URL: process.env.REACT_APP_API_URL,
  MAX_RETRIES: 3,
  TIMEOUT: 5000
};
```

Marionette 使用 **Hydra** 配置系统，功能更强大：

```yaml
# config/train.yaml
trainer:
  max_epochs: 100
  accelerator: gpu  # or cpu

data:
  root: ./data/
  name: Istanbul
  batch_size: 512
```

---

### 3.2 配置文件结构

让我们探索配置文件：

In [ ]:
import yaml
import os

# 检查配置文件是否存在
config_path = "config/train.yaml"

if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)

    print("📋 训练配置文件 (config/train.yaml):")
    print("="*50)

    # 美化打印配置
    def print_config(d, indent=0):
        for key, value in d.items():
            prefix = "  " * indent
            if isinstance(value, dict):
                print(f"{prefix}{key}:")
                print_config(value, indent + 1)
            else:
                print(f"{prefix}{key}: {value}")

    print_config(config)
else:
    print(f"⚠️  配置文件不存在: {config_path}")
    print("提示：请确保项目文件已正确加载")

⚠️  配置文件不存在: config/train.yaml
提示：请确保项目文件已正确加载


### 3.3 配置系统的工作流程

#### 前端类比

```javascript
// React: 使用 Context 管理配置
const ConfigContext = createContext(config);

// 组件中使用
const { API_URL } = useContext(ConfigContext);
```

```python
# Marionette: 使用 Hydra 管理配置
@hydra.main(config_path="config", config_name="train")
def main(cfg):
    # cfg 是一个 DictConfig 对象
    batch_size = cfg.data.batch_size
```

---

### 3.4 配置实例化流程

configs.py 提供了**工厂函数**来实例化模型：

In [ ]:
# 探索 configs.py 中的工厂函数
import sys
sys.path.append(".")

try:
    import configs
    import inspect

    print("🔧 configs.py 中的工厂函数：")
    print("="*50)

    for name, obj in inspect.getmembers(configs):
        if inspect.isfunction(obj) and not name.startswith('_'):
            # 获取函数签名
            sig = inspect.signature(obj)
            print(f"\n{name}{sig}")

            # 获取文档字符串
            if obj.__doc__:
                print(f"  📝 {obj.__doc__.strip()}")

except ImportError as e:
    print(f"⚠️  无法导入 configs: {e}")
    print("提示：请确保在项目根目录下运行")

---

## Part 4: 训练流程深度解析

### 4.1 train.py 的执行流程

#### 前端类比：React 组件生命周期

```javascript
// React 组件生命周期
function MyComponent() {
    // 1. 初始化状态
    const [data, setData] = useState(null);
    
    // 2. 副作用（加载数据）
    useEffect(() => {
        fetchData().then(setData);
    }, []);
    
    // 3. 渲染
    return <div>{data}</div>;
}
```

```python
# 训练流程
def main(config):
    # 1. 初始化（设置种子、W&B）
    set_seed(config.seed)
    wandb.init(...)
    
    # 2. 准备数据（类似 useEffect）
    datamodule = configs.instantiate_datamodule(config.data)
    datamodule.setup()
    
    # 3. 创建模型（类似组件挂载）
    tpp_model, discrete_diffusion = configs.instantiate_model(...)
    
    # 4. 训练（类似渲染 + 更新）
    trainer = pl.Trainer(...)
    trainer.fit(task, datamodule)
```

---

### 4.2 详细流程图

In [ ]:
# 用代码可视化训练流程
from rich.console import Console
from rich.table import Table
from rich import box

console = Console()

# 创建流程表
table = Table(title="训练流程详解", box=box.ROUNDED)
table.add_column("步骤", style="cyan", width=10)
table.add_column("操作", style="green", width=25)
table.add_column("前端类比", style="yellow", width=20)
table.add_column("关键点", style="white", width=30)

steps = [
    ("1", "设置随机种子", "Math.seed()", "保证结果可复现"),
    ("2", "初始化 W&B", "Analytics.init()", "记录训练指标"),
    ("3", "实例化数据模块", "useEffect()", "加载 train/test 数据"),
    ("4", "实例化模型", "<Component />", "AddThin + Transformer"),
    ("5", "实例化任务", "useReducer()", "定义训练逻辑"),
    ("6", "创建 Trainer", "ReactDOM.render()", "设置训练参数"),
    ("7", "开始训练", "render() loop", "前向 + 反向 + 更新"),
]

for step, action, analogy, key in steps:
    table.add_row(step, action, analogy, key)

console.print(table)

---

## Part 5: 实践练习

### ✅ 练习 1: 验证环境

确保所有依赖正确安装：

In [2]:
# TODO: 验证环境配置

import sys

print("🔍 环境验证检查清单：")
print("="*50)

# 检查 Python 版本
print(f"\n✓ Python 版本: {sys.version.split()[0]}")

# 检查关键包
packages = [
    ('torch', 'PyTorch'),
    ('pytorch_lightning', 'PyTorch Lightning'),
    ('numpy', 'NumPy'),
    ('pandas', 'Pandas'),
    ('hydra', 'Hydra'),
    ('wandb', 'W&B'),
]

for module, name in packages:
    try:
        __import__(module)
        print(f"✓ {name}: 已安装")
    except ImportError:
        print(f"✗ {name}: 未安装")

# 检查 GPU
import torch
if torch.cuda.is_available():
    print(f"\n✓ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("\n⚠️  GPU: 不可用（训练会很慢）")

print("\n" + "="*50)
print("环境验证完成！")

🔍 环境验证检查清单：

✓ Python 版本: 3.12.12
✓ PyTorch: 已安装
✗ PyTorch Lightning: 未安装
✓ NumPy: 已安装
✓ Pandas: 已安装
✓ Hydra: 已安装
✓ W&B: 已安装

✓ GPU: Tesla T4

环境验证完成！


### ✅ 练习 2: 探索项目结构

找到并列出每个核心模块的主要类和函数：

In [5]:
# TODO: 探索核心模块

import os

def explore_module(filepath, description):
    """探索 Python 模块的主要内容"""
    if not os.path.exists(filepath):
        return None

    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    # 简单提取类和函数定义
    import re
    classes = re.findall(r'^class (\w+)', content, re.MULTILINE)
    functions = re.findall(r'^def (\w+)', content, re.MULTILINE)

    return {
        'description': description,
        'classes': classes[:5],  # 只显示前 5 个
        'functions': functions[:5]
    }

# 探索关键模块
modules_to_explore = [
    ('datamodule.py', '数据管理和批次处理'),
    ('tasks.py', '训练任务定义'),
    ('configs.py', '配置工厂函数'),
]

print("🔍 核心模块探索：")
print("="*50)

for filepath, desc in modules_to_explore:
    result = explore_module(filepath, desc)
    if result:
        print(f"\n📄 {filepath}: {result['description']}")
        if result['classes']:
            print(f"  类: {', '.join(result['classes'])}")
        if result['functions']:
            print(f"  函数: {', '.join(result['functions'])}")
    else:
        print(f"\n⚠️  找不到文件: {filepath}")

🔍 核心模块探索：

📄 datamodule.py: 数据管理和批次处理
  类: Sequence, Batch, SequenceDataset, DataModule
  函数: pad, load_sequences

📄 tasks.py: 训练任务定义
  类: Tasks, DensityEstimation

📄 configs.py: 配置工厂函数
  函数: instantiate_datamodule, instantiate_model, instantiate_task


### ✅ 练习 3: 模块关系图

绘制项目模块的关系图（理解数据流动）：

In [6]:
# TODO: 绘制模块关系图

print("📊 Marionette 模块关系图：")
print("="*50)
print("""

    ┌─────────────────────────────────────────────┐
    │          train.py (训练入口)                 │
    └──────────────────┬──────────────────────────┘
                       │
                       ▼
    ┌─────────────────────────────────────────────┐
    │        configs.py (配置工厂)                 │
    │  • instantiate_datamodule()                  │
    │  • instantiate_model()                      │
    │  • instantiate_task()                       │
    └─────────┬───────────────┬───────────────────┘
              │               │
         ┌────┴────┐     ┌───┴────────┐
         ▼         ▼     ▼            ▼
    ┌────────┐ ┌────────┐ ┌──────────────────┐
    │  Data  │ │  TPP   │ │  Discrete        │
    │ Module │ │ Model  │ │  Diffusion       │
    └────────┘ └────────┘ └──────────────────┘
         │         │              │
         └─────────┴──────────────┘
                   │
                   ▼
            ┌─────────────┐
            │   Tasks     │
            │ (训练逻辑)   │
            └─────────────┘

""")
print("="*50)

print("\n📝 数据流动方向：")
print("""
1. train.py 启动训练
2. configs.py 创建各个组件
3. DataModule 加载和预处理数据
4. 两个模型分别处理时间和空间
5. Tasks 协调整个训练过程
""")

📊 Marionette 模块关系图：


    ┌─────────────────────────────────────────────┐
    │          train.py (训练入口)                 │
    └──────────────────┬──────────────────────────┘
                       │
                       ▼
    ┌─────────────────────────────────────────────┐
    │        configs.py (配置工厂)                 │
    │  • instantiate_datamodule()                  │
    │  • instantiate_model()                      │
    │  • instantiate_task()                       │
    └─────────┬───────────────┬───────────────────┘
              │               │
         ┌────┴────┐     ┌───┴────────┐
         ▼         ▼     ▼            ▼
    ┌────────┐ ┌────────┐ ┌──────────────────┐
    │  Data  │ │  TPP   │ │  Discrete        │
    │ Module │ │ Model  │ │  Diffusion       │
    └────────┘ └────────┘ └──────────────────┘
         │         │              │
         └─────────┴──────────────┘
                   │
                   ▼
            ┌─────────────┐
            │   Tasks  

---

## 📝 Day 1 总结

### 今天学到了什么？

#### ✅ 技术层面

1. **环境配置**
   - Colab GPU 环境设置
   - 依赖包安装

2. **项目架构**
   - 混合扩散模型设计
   - 时间与空间分离建模

3. **配置系统**
   - Hydra YAML 配置
   - 工厂函数模式

4. **训练流程**
   - 7 步训练流程
   - 模块间数据流动

#### ✅ 概念层面

| 前端概念 | AI/Python 对应 |
|---------|---------------|
| 组件生命周期 | 训练流程 |
| 配置文件 | YAML 配置 |
| 状态管理 | 数据模块 |
| 工厂函数 | 实例化 |

---

### 🎯 明天预告：Python 数据科学生态

Day 2 将学习：

1. **NumPy** - 高性能数组操作（类似 TypedArrays）
2. **Pandas** - 数据分析工具（类似 Lodash + 更多）
3. **PyTorch** - 深度学习框架（类似 TensorFlow.js）
4. **Tensor 操作** - GPU 并行计算基础

---

### 📚 课后作业

#### 必做
1. ✅ 确保环境配置成功
2. ✅ 运行所有代码单元格
3. ✅ 完成三个练习题

#### 选做
1. 📖 阅读 `resources/前端概念对照表.md`
2. 📖 查看 `docs/Marionette项目技术文档.md` 的前 3 节
3. 💡 思考：为什么轨迹生成需要分离时间和空间？

---

### ❓ 自我检查

回答以下问题，确保理解今天的重点：

1. **问题 1**: Marionette 为什么要使用混合架构（分开时间和空间）？

   <details>
   <summary>点击查看答案</summary>
   
   **答案**:
   - 时间是连续值，空间是离散值
   - 两者的统计特性不同
   - 可以使用最适合的扩散模型类型
   - 提高生成质量和训练效率
   </details>

2. **问题 2**: configs.py 中的工厂函数有什么作用？

   <details>
   <summary>点击查看答案</summary>
   
   **答案**:
   - 根据配置创建模型、数据和任务实例
   - 统一管理实例化逻辑
   - 类似前端的 Context Provider
   </details>

3. **问题 3**: train.py 的主要流程是什么？

   <details>
   <summary>点击查看答案</summary>
   
   **答案**:
   - 1. 设置种子 → 2. 初始化 W&B → 3. 加载数据
   - → 4. 创建模型 → 5. 创建任务 → 6. 设置回调
   - → 7. 开始训练
   </details>

---

**🎉 恭喜完成 Day 1！**

明天我们将深入 Python 数据科学生态，为理解核心代码打下基础。

*有问题随时查阅 `resources/` 目录下的辅助材料*